# Phase 1: Comprehensive Dataset Forensics
## Amazon ML Challenge 2026: Business Entity Resolution

### Objective:
Given business records from 3 independent sources ($S_1$: reference deduplicated, $S_2$ and $S_3$: noisy input records), identify all matching records in $S_2$ and $S_3$ for each $S_1$ entity.

### Key Evaluation Metric:
$$\text{Macro } F_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$
- **Precision-heavy**: penalizes false merges ($2\times$ weight on precision).
- **Singletons included**: predicting empty string `""` for a true singleton scores $1.0$; predicting any false match scores $0.0$.

---

In [ ]:
import os
import sys
import time
import gc
import pandas as pd
import numpy as np
from collections import Counter

# Change working directory to project root if running from notebooks directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("Working directory:", os.getcwd())
DATASET_DIR = "dataset"

## 1. Load Training and Testing Datasets
We load all 7 TSV files sequentially using memory-efficient string typing.

In [ ]:
def load_tsv(filename, split="train", usecols=None):
    filepath = os.path.join(DATASET_DIR, split, filename)
    t0 = time.time()
    df = pd.read_csv(filepath, sep="\t", dtype=str, usecols=usecols, keep_default_na=False)
    print(f"Loaded {filename} ({len(df):,} rows) in {time.time()-t0:.2f}s")
    return df

print("--- Training Data ---")
s1_train = load_tsv("train_source1.tsv", "train")
s2_train = load_tsv("train_source2.tsv", "train")
s3_train = load_tsv("train_source3.tsv", "train")
gt_train = load_tsv("train_ground_truth.tsv", "train")

print("\n--- Testing Data ---")
s1_test = load_tsv("test_source1.tsv", "test")
s2_test = load_tsv("test_source2.tsv", "test")
s3_test = load_tsv("test_source3.tsv", "test")

## 2. Dataset Scale & Search Space Analysis
Let us analyze the sheer volume of records and why blocking / candidate generation is mathematically mandatory.

In [ ]:
scale_summary = pd.DataFrame([
    {"Split": "Train", "Source 1 (Ref)": len(s1_train), "Source 2 (Noisy)": len(s2_train), "Source 3 (Noisy)": len(s3_train), "Total Candidate Target": len(s2_train) + len(s3_train)},
    {"Split": "Test", "Source 1 (Ref)": len(s1_test), "Source 2 (Noisy)": len(s2_test), "Source 3 (Noisy)": len(s3_test), "Total Candidate Target": len(s2_test) + len(s3_test)}
])

display(scale_summary)

cartesian_test_pairs = len(s1_test) * (len(s2_test) + len(s3_test))
print(f"\nTotal unconstrained pairwise comparisons in Test Set: {cartesian_test_pairs:,} (~{cartesian_test_pairs/1e12:.2f} Trillion pairs!)")
print("-> Consequence: Direct pairwise scoring is impossible. Blocking must achieve >=99.99% reduction ratio.")

## 3. Data Completeness & Missing Value Profiling

In [ ]:
def check_missing(dfs, names):
    records = []
    for df, name in zip(dfs, names):
        for col in df.columns:
            empty_cnt = ((df[col] == "") | (df[col].isna())).sum()
            empty_pct = (empty_cnt / len(df)) * 100
            records.append({"Dataset": name, "Column": col, "Total Rows": len(df), "Missing Count": empty_cnt, "Missing %": f"{empty_pct:.2f}%"})
    return pd.DataFrame(records)

display(check_missing(
    [s1_train, s2_train, s3_train, s1_test, s2_test, s3_test],
    ["Train S1", "Train S2", "Train S3", "Test S1", "Test S2", "Test S3"]
))

## 4. Text Length & Token Count Distribution
We profile character lengths and word token counts to understand entity name and address verbosity.

In [ ]:
def compute_length_stats(df, name):
    stats = []
    for col in ["business_name", "business_address"]:
        if col in df.columns:
            lengths = df[col].str.len().values
            non_empty = lengths[lengths > 0]
            tokens = df[col].str.split().str.len().values
            non_empty_tok = tokens[tokens > 0]
            stats.append({
                "Dataset": name,
                "Field": col,
                "Min Chars": non_empty.min(),
                "Mean Chars": round(non_empty.mean(), 1),
                "Median Chars": int(np.median(non_empty)),
                "P90 Chars": int(np.percentile(non_empty, 90)),
                "P95 Chars": int(np.percentile(non_empty, 95)),
                "Max Chars": non_empty.max(),
                "Mean Tokens": round(non_empty_tok.mean(), 1),
                "Median Tokens": int(np.median(non_empty_tok)),
                "P95 Tokens": int(np.percentile(non_empty_tok, 95))
            })
    return stats

all_stats = []
for df, name in zip([s1_train, s2_train, s3_train, s1_test, s2_test, s3_test], ["Train S1", "Train S2", "Train S3", "Test S1", "Test S2", "Test S3"]):
    all_stats.extend(compute_length_stats(df, name))

display(pd.DataFrame(all_stats))

## 5. Country Breakdown & Unseen France Domain Shift
Checking country distributions across training and testing splits.

In [ ]:
country_records = []
for df, name in zip([s1_train, s2_train, s3_train, s1_test, s2_test, s3_test], ["Train S1", "Train S2", "Train S3", "Test S1", "Test S2", "Test S3"]):
    counts = df["country"].value_counts()
    for c_val, cnt in counts.items():
        country_records.append({"Dataset": name, "Country": c_val, "Count": cnt, "Percentage": f"{cnt/len(df)*100:.2f}%"})

display(pd.DataFrame(country_records))
print("\nKEY INSIGHT: France represents ~15% of the test set (259.5k S1 entities) but 0% of train.")
print("Blocking & feature generation MUST treat country as an open-set string partitioned dynamically.")

## 6. Ground Truth Structure, Singletons & Distractor Analysis
Analyzing the reference matching graph in `train_ground_truth.tsv`.

In [ ]:
def parse_matches(val):
    if not val or val.strip() == "":
        return []
    return [x.strip() for x in val.split(",") if x.strip()]

gt_train["match_list"] = gt_train["matched_entity_ids"].apply(parse_matches)
gt_train["match_count"] = gt_train["match_list"].apply(len)

total_s1 = len(gt_train)
singletons = (gt_train["match_count"] == 0).sum()
multi_match = (gt_train["match_count"] > 0).sum()
total_positive_links = gt_train["match_count"].sum()

print(f"Total Reference S1 Entities in GT: {total_s1:,}")
print(f"Singletons (0 matches):            {singletons:,} ({singletons/total_s1*100:.2f}%)")
print(f"Entities with >= 1 matches:        {multi_match:,} ({multi_match/total_s1*100:.2f}%)")
print(f"Total Positive Match Links:        {total_positive_links:,}")
print(f"Mean matches per S1:               {gt_train['match_count'].mean():.2f} (Median={gt_train['match_count'].median():.0f}, Max={gt_train['match_count'].max()})")

# Histogram of match counts
hist_df = pd.DataFrame([
    {"Matches per S1": k, "S1 Count": cnt, "Percentage": f"{cnt/total_s1*100:.2f}%"}
    for k, cnt in sorted(Counter(gt_train["match_count"]).items())
])
display(hist_df.head(10))

# Source 2 vs Source 3 coverage
all_pos_ids = [m for sub in gt_train["match_list"] for m in sub]
s2_pos = set([m for m in all_pos_ids if m.startswith("S2-")])
s3_pos = set([m for m in all_pos_ids if m.startswith("S3-")])

print(f"\nSource 2 True Matches: {len(s2_pos):,} / {len(s2_train):,} ({len(s2_pos)/len(s2_train)*100:.2f}% coverage; {len(s2_train)-len(s2_pos):,} non-matching distractors)")
print(f"Source 3 True Matches: {len(s3_pos):,} / {len(s3_train):,} ({len(s3_pos)/len(s3_train)*100:.2f}% coverage; {len(s3_train)-len(s3_pos):,} non-matching distractors)")

## 7. Empirical Noise Inspection
Inspecting real match variations across languages, scripts, abbreviations, and formatting.

In [ ]:
# Sample actual matched pairs to inspect noise
s1_dict = s1_train.set_index("entity_id").to_dict("index")
s2_dict = s2_train.set_index("entity_id").to_dict("index")
s3_dict = s3_train.set_index("entity_id").to_dict("index")

sample_rows = []
for _, row in gt_train[gt_train["match_count"] > 0].head(50).iterrows():
    s1_id = row["source1_entity_id"]
    s1_info = s1_dict.get(s1_id, {})
    for m_id in row["match_list"]:
        m_info = s2_dict.get(m_id, {}) if m_id.startswith("S2-") else s3_dict.get(m_id, {})
        if m_info:
            sample_rows.append({
                "S1 ID": s1_id,
                "S1 Name": s1_info.get("business_name", ""),
                "S1 Address": s1_info.get("business_address", "")[:35],
                "Matched ID": m_id,
                "Matched Name": m_info.get("business_name", ""),
                "Matched Address": m_info.get("business_address", "")[:35],
                "Country": s1_info.get("country", "")
            })
    if len(sample_rows) >= 15:
        break

display(pd.DataFrame(sample_rows))

## 8. Summary of Findings & Next Steps (Phase 2)

1. **Multi-Script Support**: We discovered Hindi (Devanagari) and Tamil script entities in Source 2 and Source 3 matching English Latin script in Source 1. Address numbers and invariant tokens must anchor these matches.
2. **Legal Suffixes & Web Domains**: Suffixes like `LLC`, `Pvt Ltd`, `Inc`, `SARL` and web domains (`*.com`) require standardized transformations in Phase 2.
3. **Missing Address Resilience**: ~3% of target records lack addresses; model features must include `is_address_missing` flags.
4. **High Reduction Blocking**: Candidate pairs must achieve $>99.99\%$ reduction to make ML inference feasible.

**Proceeding to Phase 2: Multi-Representation Normalization Engine.**